# 01_data_prep.ipynb — NovaMart Data Preparation
**Project 2 — Version 1: Category & Sub-Category Sales Explorer**


In [8]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## Step 1 — Read & combine
Read all six monthly exports and stack them into a single DataFrame with `pd.concat`.

In [9]:
ord_1= pd.read_csv("C:\\Users\\umrrr\\Downloads\\p2_NovaMart_umar_v1\\monthly_orders\\novamart_orders_2024_01.csv")
ord_2 = pd.read_csv("C:\\Users\\umrrr\\Downloads\\p2_NovaMart_umar_v1\\monthly_orders\\novamart_orders_2024_02.csv")
ord_3 = pd.read_csv("C:\\Users\\umrrr\\Downloads\\p2_NovaMart_umar_v1\\monthly_orders\\novamart_orders_2024_03.csv")
ord_4 = pd.read_csv("C:\\Users\\umrrr\\Downloads\\p2_NovaMart_umar_v1\\monthly_orders\\novamart_orders_2024_04.csv")
ord_5 = pd.read_csv("C:\\Users\\umrrr\\Downloads\\p2_NovaMart_umar_v1\\monthly_orders\\novamart_orders_2024_05.csv")
ord_6 = pd.read_csv("C:\\Users\\umrrr\\Downloads\\p2_NovaMart_umar_v1\\monthly_orders\\novamart_orders_2024_06.csv")

orders = pd.concat([ord_1, ord_2, ord_3, ord_4, ord_5, ord_6], ignore_index=True)

print(orders.shape)
print(orders.head())

(6107, 12)
         order_id    order_date     ship_date customer_id product_id   region  \
0  NM202401461113    2024-01-27    2024-02-01      CU0049      PR522  central   
1  NM202401729507    2024-01-29    2024-01-31      CU0229      PR507  CENTRAL   
2  NM202401472052    2024-01-08    2024-01-13      CU0108      PR501     east   
3  NM202401384236    2024-01-22    2024-01-27      CU0220      PR503  CENTRAL   
4  NM202401487969  Jan 04, 2024  Jan 07, 2024      CU0213      PR501    SOUTH   

          category sub_category  quantity      sales  discount  profit  
0  office supplies      Storage         1     $41.69      0.00   19.67  
1       Technology     Machines         4    $170.76      0.00   91.36  
2       technology       Phones         1    $728.04      0.30   14.63  
3       Technology       Phones         7    $921.60      0.15     NaN  
4       technology       Phones         2  $2,080.12      0.00  653.30  


## Step 2 — De-duplicate
Some rows are fully duplicated across the exports. Drop exact duplicate rows.

In [10]:
before = len(orders)
orders = orders.drop_duplicates()
after = len(orders)
print(f'Dropped {before - after} duplicate rows')

Dropped 48 duplicate rows


## Step 3 — Fix the sales column
`sales` is stored as text like `"$1,234.50"`. Strip the `$` and `,` characters, then cast to float.

In [11]:
orders['sales'] = (
    orders['sales']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)
orders['sales'].describe()

count    6059.000000
mean     1047.621982
std      1343.071145
min         3.860000
25%       155.390000
50%       477.580000
75%      1391.190000
max      8320.480000
Name: sales, dtype: float64

## Step 4 — Parse dates

In [12]:
orders['order_date'] = pd.to_datetime(orders['order_date'], format='mixed')
orders['ship_date'] = pd.to_datetime(orders['ship_date'], format='mixed')
orders[['order_date', 'ship_date']].dtypes

order_date    datetime64[ns]
ship_date     datetime64[ns]
dtype: object

## Step 5 — Handle missing values
Blank `discount` becomes 0 (no discount applied). Blank `profit` will be recomputed after we
bring in `unit_cost` from the product lookup in Step 7.

In [13]:
orders['discount'] = orders['discount'].fillna(0)
orders.isnull().sum()

order_id          0
order_date        0
ship_date         0
customer_id       0
product_id        0
region            0
category          0
sub_category      0
quantity          0
sales             0
discount          0
profit          179
dtype: int64

## Step 6 — Load the lookup tables

In [14]:
product_lookup = pd.read_csv('data/product_lookup.csv')
customer_lookup = pd.read_csv('data/customer_lookup.csv')

product_lookup.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/product_lookup.csv'

In [ ]:
customer_lookup.head()

,customer_id,customer_name,segment,region,city
0,CU0001,Taylor Hunt,Consumer,South,Atlanta
1,CU0002,Alex Lee,Consumer,West,Seattle
2,CU0003,Riley Roy,Consumer,East,Boston
3,CU0004,Jamie Roy,Consumer,South,Atlanta
4,CU0005,Jamie Fox,Home Office,East,New York


## Step 7 — Enrich (merge)
Merge in the clean `category`, `sub_category`, `unit_cost`, `list_price` from
`product_lookup` on `product_id`, and clean `segment`, `region`, `customer_name` from
`customer_lookup` on `customer_id`. We drop the messy `region`/`category`/`sub_category`
columns that came from the raw monthly files and keep the clean lookup versions.

In [ ]:
orders = orders.drop(columns=['region', 'category', 'sub_category'])

orders = orders.merge(
    product_lookup[['product_id', 'category', 'sub_category', 'unit_cost', 'list_price']],
    on='product_id', how='left'
)

orders = orders.merge(
    customer_lookup[['customer_id', 'customer_name', 'segment', 'region']],
    on='customer_id', how='left'
)

orders.head()

,order_id,order_date,ship_date,customer_id,product_id,quantity,sales,discount,profit,category,sub_category,unit_cost,list_price,customer_name,segment,region
0,NM202401461113,2024-01-27,2024-02-01,CU0049,PR522,1,41.69,0.00,19.67,Office Supplies,Storage,22.02,41.69,Sam Bell,Corporate,Central
1,NM202401729507,2024-01-29,2024-01-31,CU0229,PR507,4,170.76,0.00,91.36,Technology,Machines,19.85,42.69,Drew Bell,Corporate,Central
2,NM202401472052,2024-01-08,2024-01-13,CU0108,PR501,1,728.04,0.30,14.63,Technology,Phones,713.41,1040.06,Sam Dean,Consumer,East
3,NM202401384236,2024-01-22,2024-01-27,CU0220,PR503,7,921.60,0.15,NaN,Technology,Phones,98.09,154.89,Drew Dean,Consumer,Central
4,NM202401487969,2024-01-04,2024-01-07,CU0213,PR501,2,2080.12,0.00,653.30,Technology,Phones,713.41,1040.06,Drew Wood,Consumer,South


## Step 8 — Recompute missing profit
Now that `unit_cost` is available, fill any still-blank `profit` using
`profit = sales - (quantity * unit_cost)`.

In [ ]:
missing_profit = orders['profit'].isnull()
print(f'Recomputing profit for {missing_profit.sum()} rows')

orders.loc[missing_profit, 'profit'] = (
    orders.loc[missing_profit, 'sales'] - (orders.loc[missing_profit, 'quantity'] * orders.loc[missing_profit, 'unit_cost'])
)
orders['profit'].isnull().sum()

Recomputing profit for 179 rows


np.int64(0)

## Step 9 — Engineer new columns
- `order_month`: month name/period the order was placed, for the monthly trend chart
- `profit_margin_pct`: profit / sales * 100
- `shipping_days`: ship_date - order_date, in days

In [ ]:
orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)
orders['profit_margin_pct'] = np.where(
    orders['sales'] != 0,
    (orders['profit'] / orders['sales']) * 100,
    0
)
orders['shipping_days'] = (orders['ship_date'] - orders['order_date']).dt.days

orders[['order_date', 'order_month', 'sales', 'profit', 'profit_margin_pct', 'shipping_days']].head()

,order_date,order_month,sales,profit,profit_margin_pct,shipping_days
0,2024-01-27,2024-01,41.69,19.67,47.181578,5
1,2024-01-29,2024-01,170.76,91.36,53.501991,2
2,2024-01-08,2024-01,728.04,14.63,2.009505,5
3,2024-01-22,2024-01,921.60,234.97,25.495877,5
4,2024-01-04,2024-01,2080.12,653.30,31.406842,3


## Step 10 — Final check & save
Quick sanity check, then save the clean file. `dashboard.py` will load this file directly.

In [ ]:
print(orders.shape)
print(orders.dtypes)
print(orders.isnull().sum())
orders.describe(include='all').T

(6059, 19)
order_id                        str
order_date           datetime64[us]
ship_date            datetime64[us]
customer_id                     str
product_id                      str
quantity                      int64
sales                       float64
discount                    float64
profit                      float64
category                        str
sub_category                    str
unit_cost                   float64
list_price                  float64
customer_name                   str
segment                         str
region                          str
order_month                     str
profit_margin_pct           float64
shipping_days                 int64
dtype: object
order_id             0
order_date           0
ship_date            0
customer_id          0
product_id           0
quantity             0
sales                0
discount             0
profit               0
category             0
sub_category         0
unit_cost            0
list_price     

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
order_id,6059,6055,NM202402126077,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_date,6059,NaN,NaN,NaN,2024-04-01 11:44:25.984486,2024-01-01 00:00:00,2024-02-16 00:00:00,2024-04-03 00:00:00,2024-05-17 00:00:00,2024-06-30 00:00:00,NaN
ship_date,6059,NaN,NaN,NaN,2024-04-04 23:43:07.555702,2024-01-01 00:00:00,2024-02-19 00:00:00,2024-04-06 00:00:00,2024-05-20 00:00:00,2024-07-07 00:00:00,NaN
customer_id,6059,240,CU0204,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_id,6059,24,PR508,275,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quantity,6059.0,NaN,NaN,NaN,4.471365,1.0,3.0,4.0,6.0,8.0,2.272453
sales,6059.0,NaN,NaN,NaN,1047.621982,3.86,155.39,477.58,1391.19,8320.48,1343.071145
discount,6059.0,NaN,NaN,NaN,0.139792,0.0,0.0,0.1,0.2,0.6,0.158433
profit,6059.0,NaN,NaN,NaN,306.561086,-2379.09,22.2,112.38,392.98,2613.2,506.137381
category,6059,3,Office Supplies,2293,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
orders.to_csv('data/novamart_clean.csv', index=False)
print('Saved data/novamart_clean.csv')

Saved data/novamart_clean.csv
